# Let's turn our voice into text, in a notebook

Shoutout to [OpenAI's Whisper](https://github.com/openai/whisper) and the
[ExecuTorch speech-to-text playground](https://executorch.ai/playground/webgpu/#/demo/speech-to-text)
for the inspiration!

Try a short recording or audio file and watch the words appear. PyTorch, audio
processing, compilation and transcription all run in your browser.

The saved result below uses the included sample clip. To try it yourself, choose
CPU or WebGPU with `DEVICE`, then select **Run → Run All Cells**. Your microphone
only starts when you click **Record microphone**.


In [2]:
import wave
from pathlib import Path

import numpy as np
import torch
from transformers import WhisperFeatureExtractor, WhisperForConditionalGeneration, WhisperTokenizer
from speech_pipeline import WhisperSpeech

DEVICE = 'cpu'
MODEL_DIR = Path('/models/whisper-base')
torch.autograd.set_multithreading_enabled(False)
print(f'PyTorch {torch.__version__} | {DEVICE}')


/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/lib/python3.14/site-packages/huggingface_hub/utils/_fixes.py:22: UserWarning: only soft file lock is available
  from filelock import BaseFileLock, FileLock, SoftFileLock, Timeout


PyTorch 2.15.0a0 | cpu


## 1. Load the model and sample clip

We're using the original Whisper base model. The first run downloads about
277 MiB of weights. Later runs reuse the verified browser cache while
browser storage remains available.

The sample is an 11-second excerpt from John F. Kennedy's inaugural address,
distributed with Whisper's tests. Your own clips can be up to 15 seconds long.


In [3]:
from whisper_weights import ensure_checkpoint

model = None
await ensure_checkpoint(MODEL_DIR)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=False,
    attn_implementation='eager').eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
extractor = WhisperFeatureExtractor.from_pretrained(MODEL_DIR, local_files_only=True)

with wave.open('jfk.wav', 'rb') as source:
    assert source.getframerate() == 16000 and source.getnchannels() == 1
    sample_audio = np.frombuffer(source.readframes(source.getnframes()), dtype='<i2').astype(np.float32) / 32768
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')
print(f'{len(sample_audio) / 16000:.1f} seconds of sample audio ready')


Loading Whisper base weights (first download: 277 MiB)...
Using verified cached weights.
72,593,920 parameters
11.0 seconds of sample audio ready


## 2. Compile and transcribe

The first transcription compiles the model for the selected device. Wait for
**Model ready**, then listen to the sample and read its transcript below.


In [5]:
if DEVICE == 'webgpu':
    from iree.runtime.webgpu import get_webgpu_device
    from iree.turbine.runtime.torch_device import TorchDevice
    gpu = TorchDevice(await get_webgpu_device(
        max_concurrent_operations=len(model.state_dict()) + 32), name='webgpu')
else:
    gpu = None
model = model.to(DEVICE)
if gpu is not None:
    await gpu.synchronize()
speech = WhisperSpeech(model, tokenizer, extractor, device=DEVICE, gpu=gpu)

print('Compiling and transcribing the sample...', flush=True)
sample_result = await speech.transcribe(sample_audio)
if sample_result['cancelled'] or sample_result['truncated'] or not sample_result['text']:
    raise RuntimeError('The sample transcription did not complete.')
print(f'Model ready on {DEVICE}.')


Compiling and transcribing the sample...


/lib/python3.14/site-packages/torch/fx/interpreter.py:177: TqdmMonitorWarning: tqdm:disabling monitor support (monitor_interval = 0) due to:
can't start new thread
  pbar = tqdm(


Model ready on cpu.


## 3. Listen, read and try your own audio

Run all cells to open the live controls. Try the sample clip, upload audio, or
click **Record microphone** and allow access. Speak, then click **Stop & transcribe**.
Recognition starts after recording stops; text appears as the model decodes it.

You can listen back, choose the spoken language, cancel decoding or save the
transcript. Audio stays in this browser.


In [6]:
from IPython.display import Audio, Markdown, display
from speech_panel import SpeechPanel

display(Audio(filename='jfk.wav'))
display(Markdown('**Sample transcript**\n\n' + sample_result['text']))

if 'recorder' in globals():
    await recorder.close()
recorder = SpeechPanel(speech.transcribe, sample='jfk.wav', device=DEVICE)


**Sample transcript**

And so my fellow Americans, ask not what your country can do for you, ask what you can do for your country.

To close the panel from Python, run `await recorder.close()` in a new cell.
Re-running its cell closes the previous instance first.

Whisper can make mistakes, especially with background noise, accents or quiet audio.
Listen back and check important text. The panel skips near-silent clips and reports
when its token limit is reached.
